# Advanced Threat Modeling Platform

A comprehensive threat modeling platform with:
- File-based attack tree data management (NO MongoDB)
- Streamlit file upload interface
- Interactive D3.js visualizations with drag/drop, zoom/pan
- CVSS v3.1 scoring
- Perplexity AI integration for threat intelligence

In [ ]:
# Install required dependencies
!pip install streamlit pandas numpy requests plotly -q

In [ ]:
import streamlit as st
import json
import pandas as pd
import numpy as np
from datetime import datetime
import requests
from typing import Dict, List, Any
import base64
from io import BytesIO

## CVSS v3.1 Calculator

Implements the complete CVSS v3.1 scoring system for vulnerability assessment.

In [ ]:
class CVSSCalculator:
    """CVSS v3.1 Score Calculator"""
    
    # Base Score Metrics
    ATTACK_VECTOR = {        'Network': 0.85,
        'Adjacent': 0.62,
        'Local': 0.55,
        'Physical': 0.2
    }
    
    ATTACK_COMPLEXITY = {
        'Low': 0.77,
        'High': 0.44
    }
    
    PRIVILEGES_REQUIRED = {
        'None': 0.85,
        'Low': 0.62,
        'High': 0.27
    }
    
    USER_INTERACTION = {
        'None': 0.85,
        'Required': 0.62
    }
    
    SCOPE = {
        'Unchanged': False,
        'Changed': True
    }
    
    IMPACT = {
        'None': 0.0,
        'Low': 0.22,
        'High': 0.56
    }
    
    def calculate_base_score(self, metrics: Dict[str, str]) -> float:
        """Calculate CVSS v3.1 base score"""
        av = self.ATTACK_VECTOR[metrics.get('AttackVector', 'Network')]
        ac = self.ATTACK_COMPLEXITY[metrics.get('AttackComplexity', 'Low')]
        pr = self.PRIVILEGES_REQUIRED[metrics.get('PrivilegesRequired', 'None')]
        ui = self.USER_INTERACTION[metrics.get('UserInteraction', 'None')]
        scope_changed = self.SCOPE[metrics.get('Scope', 'Unchanged')]
        
        c = self.IMPACT[metrics.get('Confidentiality', 'None')]
        i = self.IMPACT[metrics.get('Integrity', 'None')]
        a = self.IMPACT[metrics.get('Availability', 'None')]
        
        # Calculate ISS (Impact Sub Score)
        iss = 1 - ((1 - c) * (1 - i) * (1 - a))
        
        if iss <= 0:
            return 0.0
        
        # Calculate Impact
        if scope_changed:
            impact = 7.52 * (iss - 0.029) - 3.25 * pow(iss - 0.02, 15)
        else:
            impact = 6.42 * iss
        
        # Calculate Exploitability
        exploitability = 8.22 * av * ac * pr * ui
        
        # Calculate Base Score
        if scope_changed:
            base_score = min(1.08 * (impact + exploitability), 10.0)
        else:
            base_score = min(impact + exploitability, 10.0)
        
        return round(base_score, 1)
    
    def get_severity(self, score: float) -> str:
        """Get severity rating from CVSS score"""
        if score == 0.0:
            return 'None'
        elif score < 4.0:
            return 'Low'
        elif score < 7.0:
            return 'Medium'
        elif score < 9.0:
            return 'High'
        else:
            return 'Critical'

## Perplexity AI Integration

Integration with Perplexity AI for threat intelligence and analysis.

In [ ]:
class PerplexityAI:
    """Perplexity AI Integration for Threat Intelligence"""
    
    def __init__(self, api_key: str = None):
        self.api_key = api_key or st.session_state.get('perplexity_api_key', '')
        self.api_url = 'https://api.perplexity.ai/chat/completions'
    
    def analyze_threat(self, threat_description: str) -> Dict[str, Any]:
        """Analyze a threat using Perplexity AI"""
        if not self.api_key:
            return {'error': 'API key not configured'}
        
        try:
            headers = {
                'Authorization': f'Bearer {self.api_key}',
                'Content-Type': 'application/json'
            }
            
            payload = {
                'model': 'llama-3.1-sonar-small-128k-online',
                'messages': [
                    {
                        'role': 'system',
                        'content': 'You are a cybersecurity expert analyzing threats and vulnerabilities.'
                    },
                    {
                        'role': 'user',
                        'content': f'Analyze this security threat and provide: 1) Attack vectors, 2) Potential impact, 3) Mitigation strategies. Threat: {threat_description}'
                    }
                ]
            }
            
            response = requests.post(self.api_url, headers=headers, json=payload, timeout=30)
            response.raise_for_status()
            
            result = response.json()
            return {
                'analysis': result['choices'][0]['message']['content'],
                'success': True
            }
        except Exception as e:
            return {'error': str(e), 'success': False}
    
    def suggest_mitigations(self, vulnerability: str, cvss_score: float) -> List[str]:
        """Get mitigation suggestions for a vulnerability"""
        if not self.api_key:
            return ['Configure Perplexity API key to get AI-powered suggestions']
        
        try:
            headers = {
                'Authorization': f'Bearer {self.api_key}',
                'Content-Type': 'application/json'
            }
            
            payload = {
                'model': 'llama-3.1-sonar-small-128k-online',
                'messages': [
                    {
                        'role': 'user',
                        'content': f'Provide 5 specific mitigation strategies for: {vulnerability} (CVSS: {cvss_score}). Return as numbered list.'
                    }
                ]
            }
            
            response = requests.post(self.api_url, headers=headers, json=payload, timeout=30)
            response.raise_for_status()
            
            result = response.json()
            content = result['choices'][0]['message']['content']
            
            # Parse numbered list
            mitigations = [line.strip() for line in content.split('\n') if line.strip() and any(c.isdigit() for c in line[:3])]
            return mitigations[:5]
        except:
            return ['Error fetching AI suggestions']

## Attack Tree Visualization with D3.js

Interactive attack tree visualization with draggable nodes, tooltips, zoom/pan, and filtering.

In [ ]:
def generate_d3_visualization(tree_data: Dict) -> str:
    """Generate interactive D3.js visualization HTML"""
    
    tree_json = json.dumps(tree_data)
    
    html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <script src="https://d3js.org/d3.v7.min.js"></script>
        <style>
            body {{ margin: 0; font-family: Arial, sans-serif; }}
            #tree-container {{ width: 100%; height: 800px; border: 1px solid #ccc; }}
            .node {{ cursor: pointer; }}
            .node circle {{
                fill: #fff;
                stroke: steelblue;
                stroke-width: 3px;
            }}
            .node.selected circle {{
                stroke: #ff6b6b;
                stroke-width: 4px;
                fill: #ffe0e0;
            }}
            .node.critical circle {{ stroke: #dc3545; }}
            .node.high circle {{ stroke: #fd7e14; }}
            .node.medium circle {{ stroke: #ffc107; }}
            .node.low circle {{ stroke: #28a745; }}
            .node text {{
                font-size: 12px;
                font-family: Arial, sans-serif;
            }}
            .link {{
                fill: none;
                stroke: #ccc;
                stroke-width: 2px;
            }}
            .link.highlighted {{
                stroke: #ff6b6b;
                stroke-width: 3px;
            }}
            .tooltip {{
                position: absolute;
                padding: 10px;
                background: rgba(0, 0, 0, 0.8);
                color: white;
                border-radius: 5px;
                pointer-events: none;
                font-size: 12px;
                z-index: 1000;
            }}
            .controls {{
                padding: 10px;
                background: #f8f9fa;
                border-bottom: 1px solid #dee2e6;
            }}
            .controls button {{
                margin: 0 5px;
                padding: 8px 15px;
                border: none;
                border-radius: 4px;
                background: #007bff;
                color: white;
                cursor: pointer;
            }}
            .controls button:hover {{
                background: #0056b3;
            }}
            .controls select {{
                margin: 0 5px;
                padding: 8px;
                border: 1px solid #ced4da;
                border-radius: 4px;
            }}
            .detail-panel {{
                position: fixed;
                right: 10px;
                top: 70px;
                width: 300px;
                padding: 15px;
                background: white;
                border: 1px solid #dee2e6;
                border-radius: 5px;
                box-shadow: 0 2px 10px rgba(0,0,0,0.1);
                display: none;
                max-height: 500px;
                overflow-y: auto;
            }}
            .detail-panel h3 {{
                margin-top: 0;
                color: #333;
            }}
            .detail-panel .close-btn {{
                float: right;
                cursor: pointer;
                font-size: 20px;
                color: #999;
            }}
        </style>
    </head>
    <body>
        <div class="controls">
            <button onclick="resetLayout()">Reset Layout</button>
            <button onclick="resetZoom()">Reset Zoom</button>
            <label>Filter by Severity:</label>
            <select id="severity-filter" onchange="filterBySeverity()">
                <option value="all">All</option>
                <option value="critical">Critical</option>
                <option value="high">High</option>
                <option value="medium">Medium</option>
                <option value="low">Low</option>
            </select>
            <button onclick="highlightCriticalPath()">Highlight Critical Path</button>
        </div>
        <div id="tree-container"></div>
        <div class="tooltip" id="tooltip"></div>
        <div class="detail-panel" id="detail-panel">
            <span class="close-btn" onclick="closeDetailPanel()">×</span>
            <div id="detail-content"></div>
        </div>
        
        <script>
            const treeData = {tree_json};
            
            const width = document.getElementById('tree-container').clientWidth;
            const height = 800;
            
            const svg = d3.select('#tree-container')
                .append('svg')
                .attr('width', width)
                .attr('height', height);
            
            const g = svg.append('g')
                .attr('transform', 'translate(50, 50)');
            
            // Zoom behavior
            const zoom = d3.zoom()
                .scaleExtent([0.1, 3])
                .on('zoom', (event) => {{
                    g.attr('transform', event.transform);
                }});
            
            svg.call(zoom);
            
            // Tree layout
            const treeLayout = d3.tree()
                .size([height - 100, width - 200]);
            
            const root = d3.hierarchy(treeData);
            treeLayout(root);
            
            let selectedNode = null;
            
            function update() {{
                // Links
                const links = g.selectAll('.link')
                    .data(root.links())
                    .join('path')
                    .attr('class', 'link')
                    .attr('d', d3.linkHorizontal()
                        .x(d => d.y)
                        .y(d => d.x));
                
                // Nodes
                const nodes = g.selectAll('.node')
                    .data(root.descendants())
                    .join('g')
                    .attr('class', d => {{
                        const severity = (d.data.severity || 'low').toLowerCase();
                        return `node ${{severity}}`;
                    }})
                    .attr('transform', d => `translate(${{d.y}},${{d.x}})`)
                    .call(d3.drag()
                        .on('start', dragStarted)
                        .on('drag', dragged)
                        .on('end', dragEnded))
                    .on('click', (event, d) => {{
                        event.stopPropagation();
                        selectNode(d);
                    }})
                    .on('mouseover', (event, d) => {{
                        showTooltip(event, d);
                    }})
                    .on('mouseout', hideTooltip);
                
                nodes.append('circle')
                    .attr('r', 8);
                
                nodes.append('text')
                    .attr('dy', '.31em')
                    .attr('x', d => d.children ? -12 : 12)
                    .style('text-anchor', d => d.children ? 'end' : 'start')
                    .text(d => d.data.name);
            }}
            
            function dragStarted(event, d) {{
                d3.select(this).raise();
            }}
            
            function dragged(event, d) {{
                d.x = event.y;
                d.y = event.x;
                d3.select(this).attr('transform', `translate(${{d.y}},${{d.x}})`);
                
                // Update connected links
                g.selectAll('.link')
                    .attr('d', d3.linkHorizontal()
                        .x(d => d.y)
                        .y(d => d.x));
            }}
            
            function dragEnded(event, d) {{
                // Node position is updated
            }}
            
            function selectNode(d) {{
                if (selectedNode) {{
                    d3.selectAll('.node').classed('selected', false);
                }}
                selectedNode = d;
                d3.select(event.target.parentNode).classed('selected', true);
                showDetailPanel(d);
            }}
            
            function showDetailPanel(d) {{
                const panel = document.getElementById('detail-panel');
                const content = document.getElementById('detail-content');
                
                content.innerHTML = `
                    <h3>${{d.data.name}}</h3>
                    <p><strong>Description:</strong> ${{d.data.description || 'N/A'}}</p>
                    <p><strong>CVSS Score:</strong> ${{d.data.cvss_score || 'N/A'}}</p>
                    <p><strong>Severity:</strong> ${{d.data.severity || 'N/A'}}</p>
                    <p><strong>Attack Vector:</strong> ${{d.data.attack_vector || 'N/A'}}</p>
                    <p><strong>Likelihood:</strong> ${{d.data.likelihood || 'N/A'}}</p>
                    <p><strong>Impact:</strong> ${{d.data.impact || 'N/A'}}</p>
                `;
                
                panel.style.display = 'block';
            }}
            
            function closeDetailPanel() {{
                document.getElementById('detail-panel').style.display = 'none';
                d3.selectAll('.node').classed('selected', false);
                selectedNode = null;
            }}
            
            function showTooltip(event, d) {{
                const tooltip = document.getElementById('tooltip');
                tooltip.innerHTML = `
                    <strong>${{d.data.name}}</strong><br>
                    CVSS: ${{d.data.cvss_score || 'N/A'}}<br>
                    Severity: ${{d.data.severity || 'N/A'}}
                `;
                tooltip.style.display = 'block';
                tooltip.style.left = (event.pageX + 10) + 'px';
                tooltip.style.top = (event.pageY + 10) + 'px';
            }}
            
            function hideTooltip() {{
                document.getElementById('tooltip').style.display = 'none';
            }}
            
            function resetLayout() {{
                treeLayout(root);
                update();
            }}
            
            function resetZoom() {{
                svg.transition()
                    .duration(750)
                    .call(zoom.transform, d3.zoomIdentity);
            }}
            
            function filterBySeverity() {{
                const severity = document.getElementById('severity-filter').value;
                
                g.selectAll('.node')
                    .style('opacity', d => {{
                        if (severity === 'all') return 1;
                        return (d.data.severity || '').toLowerCase() === severity ? 1 : 0.2;
                    }});
                
                g.selectAll('.link')
                    .style('opacity', d => {{
                        if (severity === 'all') return 1;
                        const sourceSev = (d.source.data.severity || '').toLowerCase();
                        const targetSev = (d.target.data.severity || '').toLowerCase();
                        return (sourceSev === severity || targetSev === severity) ? 1 : 0.2;
                    }});
            }}
            
            function highlightCriticalPath() {{
                // Find nodes with critical or high severity
                const criticalNodes = root.descendants().filter(d => {{
                    const sev = (d.data.severity || '').toLowerCase();
                    return sev === 'critical' || sev === 'high';
                }});
                
                // Reset highlighting
                g.selectAll('.link').classed('highlighted', false);
                g.selectAll('.node').style('opacity', 0.3);
                
                // Highlight critical nodes and their paths
                criticalNodes.forEach(node => {{
                    // Highlight node
                    g.selectAll('.node')
                        .filter(d => d === node)
                        .style('opacity', 1);
                    
                    // Highlight path to root
                    let current = node;
                    while (current.parent) {{
                        g.selectAll('.link')
                            .filter(d => d.target === current)
                            .classed('highlighted', true);
                        
                        g.selectAll('.node')
                            .filter(d => d === current.parent)
                            .style('opacity', 1);
                        
                        current = current.parent;
                    }}
                }});
            }}
            
            // Initialize
            update();
        </script>
    </body>
    </html>
    """
    
    return html

## Streamlit Application

Main Streamlit application with file upload, visualization, and analysis tabs.

In [ ]:
def main():
    st.set_page_config(
        page_title="Advanced Threat Modeling Platform",
        page_icon="🛡️",
        layout="wide"
    )
    
    st.title("🛡️ Advanced Threat Modeling Platform")
    st.markdown("""
    A comprehensive threat modeling platform with file-based workflow, interactive visualizations, 
    CVSS v3.1 scoring, and AI-powered threat intelligence.
    """)
    
    # Initialize session state
    if 'attack_tree' not in st.session_state:
        st.session_state.attack_tree = None
    if 'cvss_calculator' not in st.session_state:
        st.session_state.cvss_calculator = CVSSCalculator()
    
    # Sidebar configuration
    with st.sidebar:
        st.header("⚙️ Configuration")
        api_key = st.text_input(
            "Perplexity API Key", 
            type="password",
            help="Enter your Perplexity AI API key for threat intelligence"
        )
        if api_key:
            st.session_state.perplexity_api_key = api_key
        
        st.markdown("---")
        st.markdown("### 📚 Documentation")
        st.markdown("""
        **File-Based Workflow:**
        1. Upload attack tree JSON file
        2. Visualize with interactive D3.js
        3. Analyze with CVSS scoring
        4. Get AI-powered insights
        
        **No Database Required!**
        All data is managed through file uploads and downloads.
        """)
    
    # Main tabs
    tabs = st.tabs(["📤 Upload", "🌳 Visualization", "📊 Analysis", "🤖 AI Intelligence", "📖 CVSS Calculator"])
    
    # Tab 1: File Upload
    with tabs[0]:
        st.header("Upload Attack Tree Data")
        st.markdown("""
        Upload a JSON file containing your attack tree structure. The file should follow this format:
        ```json
        {
          "name": "Root Attack",
          "description": "Main attack objective",
          "cvss_score": 9.8,
          "severity": "Critical",
          "attack_vector": "Network",
          "likelihood": "High",
          "impact": "High",
          "children": []
        }
        ```
        """)
        
        uploaded_file = st.file_uploader(
            "Choose an attack tree JSON file",
            type=['json'],
            help="Upload a JSON file with attack tree structure"
        )
        
        if uploaded_file is not None:
            try:
                tree_data = json.load(uploaded_file)
                st.session_state.attack_tree = tree_data
                st.success("✅ Attack tree loaded successfully!")
                
                # Display tree summary
                st.subheader("Tree Summary")
                col1, col2, col3 = st.columns(3)
                
                def count_nodes(node):
                    count = 1
                    if 'children' in node:
                        for child in node['children']:
                            count += count_nodes(child)
                    return count
                
                def get_max_depth(node, depth=0):
                    if 'children' not in node or not node['children']:
                        return depth
                    return max(get_max_depth(child, depth + 1) for child in node['children'])
                
                with col1:
                    st.metric("Total Nodes", count_nodes(tree_data))
                with col2:
                    st.metric("Max Depth", get_max_depth(tree_data))
                with col3:
                    st.metric("Root CVSS", tree_data.get('cvss_score', 'N/A'))
                
                # Show tree structure
                with st.expander("View Raw Data"):
                    st.json(tree_data)
                
            except Exception as e:
                st.error(f"❌ Error loading file: {str(e)}")
        
        # Sample data download
        st.markdown("---")
        st.subheader("Need a sample file?")
        
        sample_tree = {
            "name": "Web Application Attack",
            "description": "Compromise web application and steal user data",
            "cvss_score": 9.8,
            "severity": "Critical",
            "attack_vector": "Network",
            "likelihood": "High",
            "impact": "High",
            "children": [
                {
                    "name": "SQL Injection",
                    "description": "Exploit SQL injection vulnerability",
                    "cvss_score": 9.8,
                    "severity": "Critical",
                    "attack_vector": "Network",
                    "likelihood": "High",
                    "impact": "High",
                    "children": [
                        {
                            "name": "Bypass Authentication",
                            "description": "Use SQL injection to bypass login",
                            "cvss_score": 9.1,
                            "severity": "Critical",
                            "attack_vector": "Network",
                            "likelihood": "Medium",
                            "impact": "High"
                        },
                        {
                            "name": "Extract Database",
                            "description": "Use UNION queries to extract data",
                            "cvss_score": 8.6,
                            "severity": "High",
                            "attack_vector": "Network",
                            "likelihood": "Medium",
                            "impact": "High"
                        }
                    ]
                },
                {
                    "name": "XSS Attack",
                    "description": "Cross-site scripting to steal session tokens",
                    "cvss_score": 7.1,
                    "severity": "High",
                    "attack_vector": "Network",
                    "likelihood": "High",
                    "impact": "Medium",
                    "children": [
                        {
                            "name": "Stored XSS",
                            "description": "Inject malicious script in stored data",
                            "cvss_score": 7.1,
                            "severity": "High",
                            "attack_vector": "Network",
                            "likelihood": "Medium",
                            "impact": "Medium"
                        },
                        {
                            "name": "Reflected XSS",
                            "description": "Craft malicious URL with XSS payload",
                            "cvss_score": 6.1,
                            "severity": "Medium",
                            "attack_vector": "Network",
                            "likelihood": "Medium",
                            "impact": "Low"
                        }
                    ]
                },
                {
                    "name": "Brute Force Attack",
                    "description": "Attempt to guess credentials",
                    "cvss_score": 5.3,
                    "severity": "Medium",
                    "attack_vector": "Network",
                    "likelihood": "Low",
                    "impact": "High",
                    "children": [
                        {
                            "name": "Dictionary Attack",
                            "description": "Use common password list",
                            "cvss_score": 5.3,
                            "severity": "Medium",
                            "attack_vector": "Network",
                            "likelihood": "Low",
                            "impact": "Medium"
                        }
                    ]
                }
            ]
        }
        
        sample_json = json.dumps(sample_tree, indent=2)
        st.download_button(
            label="Download Sample Attack Tree",
            data=sample_json,
            file_name="sample_attack_tree.json",
            mime="application/json"
        )
    
    # Tab 2: Visualization
    with tabs[1]:
        st.header("Interactive Attack Tree Visualization")
        
        if st.session_state.attack_tree is None:
            st.warning("⚠️ Please upload an attack tree file in the Upload tab first.")
        else:
            st.markdown("""
            **Features:**
            - 🖱️ **Drag nodes** to rearrange the tree
            - 🔍 **Zoom and pan** to explore large trees
            - 👆 **Click nodes** to see details
            - 🎯 **Hover** to see quick CVSS info
            - 🔴 **Filter** by severity level
            - ⚡ **Highlight** critical attack paths
            - 🔄 **Reset** to restore original layout
            """)
            
            # Generate and display visualization
            html_content = generate_d3_visualization(st.session_state.attack_tree)
            st.components.v1.html(html_content, height=850, scrolling=False)
    
    # Tab 3: Analysis
    with tabs[2]:
        st.header("Attack Tree Analysis")
        
        if st.session_state.attack_tree is None:
            st.warning("⚠️ Please upload an attack tree file in the Upload tab first.")
        else:
            tree = st.session_state.attack_tree
            
            # Collect all nodes
            all_nodes = []
            def collect_nodes(node):
                all_nodes.append(node)
                if 'children' in node:
                    for child in node['children']:
                        collect_nodes(child)
            
            collect_nodes(tree)
            
            # Statistics
            st.subheader("📊 Threat Statistics")
            col1, col2, col3, col4 = st.columns(4)
            
            severity_counts = {}
            for node in all_nodes:
                sev = node.get('severity', 'Unknown')
                severity_counts[sev] = severity_counts.get(sev, 0) + 1
            
            with col1:
                st.metric("Critical Threats", severity_counts.get('Critical', 0))
            with col2:
                st.metric("High Threats", severity_counts.get('High', 0))
            with col3:
                st.metric("Medium Threats", severity_counts.get('Medium', 0))
            with col4:
                st.metric("Low Threats", severity_counts.get('Low', 0))
            
            # Top threats
            st.subheader("🔴 Top Threats by CVSS Score")
            threats_df = pd.DataFrame([
                {
                    'Name': node.get('name', 'Unknown'),
                    'CVSS Score': node.get('cvss_score', 0),
                    'Severity': node.get('severity', 'Unknown'),
                    'Attack Vector': node.get('attack_vector', 'Unknown'),
                    'Likelihood': node.get('likelihood', 'Unknown'),
                    'Impact': node.get('impact', 'Unknown')
                }
                for node in all_nodes
            ])
            
            threats_df = threats_df.sort_values('CVSS Score', ascending=False)
            st.dataframe(threats_df, use_container_width=True)
            
            # Export analysis
            st.markdown("---")
            st.subheader("💾 Export Analysis")
            
            csv = threats_df.to_csv(index=False)
            st.download_button(
                label="Download Analysis as CSV",
                data=csv,
                file_name=f"threat_analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv",
                mime="text/csv"
            )
    
    # Tab 4: AI Intelligence
    with tabs[3]:
        st.header("🤖 AI-Powered Threat Intelligence")
        
        if 'perplexity_api_key' not in st.session_state or not st.session_state.perplexity_api_key:
            st.warning("⚠️ Please configure your Perplexity API key in the sidebar to use AI features.")
        else:
            ai = PerplexityAI(st.session_state.perplexity_api_key)
            
            st.markdown("""
            Use AI to analyze threats, get mitigation strategies, and receive intelligent insights.
            """)
            
            # Threat analysis
            st.subheader("Analyze a Threat")
            threat_input = st.text_area(
                "Describe the threat or vulnerability:",
                placeholder="e.g., SQL injection vulnerability in user login form",
                height=100
            )
            
            if st.button("🔍 Analyze Threat", type="primary"):
                if threat_input:
                    with st.spinner("Analyzing with AI..."):
                        result = ai.analyze_threat(threat_input)
                        
                        if result.get('success'):
                            st.success("Analysis complete!")
                            st.markdown(result['analysis'])
                        else:
                            st.error(f"Error: {result.get('error', 'Unknown error')}")
                else:
                    st.warning("Please enter a threat description.")
            
            # Mitigation suggestions
            st.markdown("---")
            st.subheader("Get Mitigation Strategies")
            
            col1, col2 = st.columns([3, 1])
            with col1:
                vuln_input = st.text_input(
                    "Vulnerability:",
                    placeholder="e.g., Cross-Site Scripting (XSS)"
                )
            with col2:
                cvss_input = st.number_input(
                    "CVSS Score:",
                    min_value=0.0,
                    max_value=10.0,
                    value=7.5,
                    step=0.1
                )
            
            if st.button("💡 Get Mitigations", type="primary"):
                if vuln_input:
                    with st.spinner("Fetching AI suggestions..."):
                        mitigations = ai.suggest_mitigations(vuln_input, cvss_input)
                        
                        st.subheader("Recommended Mitigations:")
                        for mitigation in mitigations:
                            st.markdown(f"- {mitigation}")
                else:
                    st.warning("Please enter a vulnerability description.")
    
    # Tab 5: CVSS Calculator
    with tabs[4]:
        st.header("📖 CVSS v3.1 Calculator")
        st.markdown("""
        Calculate CVSS v3.1 scores for your threats. Select the appropriate metrics below.
        """)
        
        calc = st.session_state.cvss_calculator
        
        col1, col2 = st.columns(2)
        
        with col1:
            st.subheader("Base Metrics")
            
            attack_vector = st.selectbox(
                "Attack Vector (AV)",
                options=list(calc.ATTACK_VECTOR.keys()),
                help="How the vulnerability is exploited"
            )
            
            attack_complexity = st.selectbox(
                "Attack Complexity (AC)",
                options=list(calc.ATTACK_COMPLEXITY.keys()),
                help="Complexity of the attack"
            )
            
            privileges_required = st.selectbox(
                "Privileges Required (PR)",
                options=list(calc.PRIVILEGES_REQUIRED.keys()),
                help="Level of privileges needed"
            )
            
            user_interaction = st.selectbox(
                "User Interaction (UI)",
                options=list(calc.USER_INTERACTION.keys()),
                help="Whether user interaction is required"
            )
        
        with col2:
            st.subheader("Impact Metrics")
            
            scope = st.selectbox(
                "Scope (S)",
                options=list(calc.SCOPE.keys()),
                help="Whether the vulnerability affects resources beyond its security scope"
            )
            
            confidentiality = st.selectbox(
                "Confidentiality Impact (C)",
                options=list(calc.IMPACT.keys()),
                help="Impact on confidentiality"
            )
            
            integrity = st.selectbox(
                "Integrity Impact (I)",
                options=list(calc.IMPACT.keys()),
                help="Impact on integrity"
            )
            
            availability = st.selectbox(
                "Availability Impact (A)",
                options=list(calc.IMPACT.keys()),
                help="Impact on availability"
            )
        
        # Calculate button
        if st.button("🔢 Calculate CVSS Score", type="primary"):
            metrics = {
                'AttackVector': attack_vector,
                'AttackComplexity': attack_complexity,
                'PrivilegesRequired': privileges_required,
                'UserInteraction': user_interaction,
                'Scope': scope,
                'Confidentiality': confidentiality,
                'Integrity': integrity,
                'Availability': availability
            }
            
            score = calc.calculate_base_score(metrics)
            severity = calc.get_severity(score)
            
            st.markdown("---")
            st.subheader("Results")
            
            col1, col2 = st.columns(2)
            with col1:
                st.metric("CVSS v3.1 Base Score", score)
            with col2:
                severity_color = {
                    'None': '🟢',
                    'Low': '🟡',
                    'Medium': '🟠',
                    'High': '🔴',
                    'Critical': '🔴'
                }
                st.metric("Severity", f"{severity_color.get(severity, '')} {severity}")
            
            st.info(f"""
            **CVSS Vector String:**
            CVSS:3.1/AV:{attack_vector[0]}/AC:{attack_complexity[0]}/PR:{privileges_required[0]}/
            UI:{user_interaction[0]}/S:{scope[0]}/C:{confidentiality[0]}/I:{integrity[0]}/A:{availability[0]}
            """)

if __name__ == "__main__":
    main()

## Running the Application

To run the Streamlit application, execute the cell below or run in terminal:
```bash
streamlit run app.py
```

Make sure to:
1. Install all dependencies from requirements.txt
2. Configure your Perplexity API key in the sidebar
3. Upload an attack tree JSON file or use the sample provided


In [ ]:
# Run the Streamlit app (only works in appropriate environment)
# !streamlit run app.py